# 01 — Synthetic Training Data Generator
## MangoSense XGBoost Symptom Classifier

**Purpose:** Generate synthetic labeled training data for the XGBoost symptom classifier used in MangoSense mango disease detection.

Since real farmer-reported symptom data is limited during early deployment, this notebook generates synthetic samples using **known disease–symptom probability associations** derived from plant pathology literature.

Each generated row represents one observation: a set of symptoms reported by a farmer paired with the ground-truth disease label.

**Outputs:**
- `leaf_training_data.csv` — 750 rows (150 per leaf disease × 5 classes)
- `fruit_training_data.csv` — 300 rows (150 per fruit disease × 2 classes)
- `all_training_data.csv` — combined dataset
- `sample_data_preview.json` — 5 examples per disease for quick inspection

**Next step:** Upload the CSV files to **Notebook 02** for XGBoost model training.

In [ ]:
!pip install pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import json
import random
import os
from pathlib import Path

print("Imports OK")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

## Vocabulary Definition

These are the canonical symptom strings and disease labels used across MangoSense.
The **order of LEAF_SYMPTOMS and FRUIT_SYMPTOMS is critical** — it determines feature indices for the XGBoost model.

In [ ]:
# ── Leaf vocabulary ──────────────────────────────────────────────────────────
LEAF_SYMPTOMS = [
    'dark_spots_brown',            # index 0
    'dark_spots_with_yellow_halo', # index 1
    'concentric_rings_on_lesion',  # index 2
    'black_specks_in_lesion',      # index 3
    'pink_spore_masses',           # index 4
    'lesions_enlarge_rapidly',     # index 5
    'spots_with_irregular_border', # index 6
    'twig_dieback_from_tip',       # index 7
    'canker_lesions_on_twig',      # index 8
    'bark_cracking',               # index 9
    'wilting_shoot_tips',          # index 10
    'sparse_foliage',              # index 11
    'white_powder_coating',        # index 12
    'leaf_distortion',             # index 13
    'premature_leaf_drop',         # index 14
    'black_sooty_coating',         # index 15
    'sooty_deposit_wiped_off',     # index 16
    'yellow_discoloration',        # index 17
    'brown_leaf_margins',          # index 18
    'brown_leaf_tips',             # index 19
    'water_soaked_lesions',        # index 20
    'leaf_curling',                # index 21
]

LEAF_DISEASES = ['Anthracnose', 'Die Back', 'Healthy', 'Powdery Mildew', 'Sooty Mold']

# ── Fruit vocabulary ─────────────────────────────────────────────────────────
FRUIT_SYMPTOMS = [
    'black_sunken_lesions',        # index 0
    'brown_patches_spreading',     # index 1
    'surface_cracks_radiating',    # index 2
    'pink_spore_masses_on_lesion', # index 3
    'lesion_ring_pattern',         # index 4
    'soft_rot_spreading',          # index 5
    'stem_end_rot',                # index 6
    'shriveling_of_fruit',         # index 7
    'premature_fruit_drop',        # index 8
    'white_powder_on_fruit',       # index 9
    'black_sooty_deposit_on_fruit',# index 10
    'fruit_discoloration',         # index 11
]

FRUIT_DISEASES = ['Anthracnose', 'Healthy']

print(f"Leaf symptoms : {len(LEAF_SYMPTOMS)} features")
print(f"Leaf diseases : {LEAF_DISEASES}")
print(f"Fruit symptoms: {len(FRUIT_SYMPTOMS)} features")
print(f"Fruit diseases: {FRUIT_DISEASES}")

In [ ]:
# ── Disease–symptom probability matrices ─────────────────────────────────────
# P(symptom present | disease)  — based on plant pathology literature
# Keys must match LEAF_DISEASES / FRUIT_DISEASES exactly.

LEAF_SYMPTOM_PROBS = {
    'Anthracnose': {
        'dark_spots_brown': 0.95,
        'dark_spots_with_yellow_halo': 0.75,
        'concentric_rings_on_lesion': 0.70,
        'black_specks_in_lesion': 0.65,
        'pink_spore_masses': 0.50,
        'lesions_enlarge_rapidly': 0.80,
        'spots_with_irregular_border': 0.60,
        'twig_dieback_from_tip': 0.10,
        'canker_lesions_on_twig': 0.05,
        'bark_cracking': 0.05,
        'wilting_shoot_tips': 0.15,
        'sparse_foliage': 0.10,
        'white_powder_coating': 0.00,
        'leaf_distortion': 0.05,
        'premature_leaf_drop': 0.40,
        'black_sooty_coating': 0.05,
        'sooty_deposit_wiped_off': 0.00,
        'yellow_discoloration': 0.30,
        'brown_leaf_margins': 0.20,
        'brown_leaf_tips': 0.15,
        'water_soaked_lesions': 0.10,
        'leaf_curling': 0.05,
    },
    'Die Back': {
        'dark_spots_brown': 0.10,
        'dark_spots_with_yellow_halo': 0.05,
        'concentric_rings_on_lesion': 0.00,
        'black_specks_in_lesion': 0.10,
        'pink_spore_masses': 0.05,
        'lesions_enlarge_rapidly': 0.20,
        'spots_with_irregular_border': 0.10,
        'twig_dieback_from_tip': 0.95,
        'canker_lesions_on_twig': 0.85,
        'bark_cracking': 0.80,
        'wilting_shoot_tips': 0.90,
        'sparse_foliage': 0.75,
        'white_powder_coating': 0.00,
        'leaf_distortion': 0.10,
        'premature_leaf_drop': 0.55,
        'black_sooty_coating': 0.05,
        'sooty_deposit_wiped_off': 0.00,
        'yellow_discoloration': 0.30,
        'brown_leaf_margins': 0.40,
        'brown_leaf_tips': 0.85,
        'water_soaked_lesions': 0.05,
        'leaf_curling': 0.10,
    },
    'Healthy': {
        'dark_spots_brown': 0.02,
        'dark_spots_with_yellow_halo': 0.01,
        'concentric_rings_on_lesion': 0.00,
        'black_specks_in_lesion': 0.01,
        'pink_spore_masses': 0.00,
        'lesions_enlarge_rapidly': 0.00,
        'spots_with_irregular_border': 0.01,
        'twig_dieback_from_tip': 0.01,
        'canker_lesions_on_twig': 0.00,
        'bark_cracking': 0.00,
        'wilting_shoot_tips': 0.02,
        'sparse_foliage': 0.02,
        'white_powder_coating': 0.00,
        'leaf_distortion': 0.02,
        'premature_leaf_drop': 0.03,
        'black_sooty_coating': 0.00,
        'sooty_deposit_wiped_off': 0.00,
        'yellow_discoloration': 0.05,
        'brown_leaf_margins': 0.03,
        'brown_leaf_tips': 0.04,
        'water_soaked_lesions': 0.01,
        'leaf_curling': 0.02,
    },
    'Powdery Mildew': {
        'dark_spots_brown': 0.05,
        'dark_spots_with_yellow_halo': 0.05,
        'concentric_rings_on_lesion': 0.00,
        'black_specks_in_lesion': 0.00,
        'pink_spore_masses': 0.00,
        'lesions_enlarge_rapidly': 0.10,
        'spots_with_irregular_border': 0.05,
        'twig_dieback_from_tip': 0.00,
        'canker_lesions_on_twig': 0.00,
        'bark_cracking': 0.00,
        'wilting_shoot_tips': 0.10,
        'sparse_foliage': 0.20,
        'white_powder_coating': 0.95,
        'leaf_distortion': 0.80,
        'premature_leaf_drop': 0.65,
        'black_sooty_coating': 0.00,
        'sooty_deposit_wiped_off': 0.00,
        'yellow_discoloration': 0.40,
        'brown_leaf_margins': 0.15,
        'brown_leaf_tips': 0.10,
        'water_soaked_lesions': 0.00,
        'leaf_curling': 0.70,
    },
    'Sooty Mold': {
        'dark_spots_brown': 0.05,
        'dark_spots_with_yellow_halo': 0.00,
        'concentric_rings_on_lesion': 0.00,
        'black_specks_in_lesion': 0.00,
        'pink_spore_masses': 0.00,
        'lesions_enlarge_rapidly': 0.00,
        'spots_with_irregular_border': 0.00,
        'twig_dieback_from_tip': 0.00,
        'canker_lesions_on_twig': 0.00,
        'bark_cracking': 0.00,
        'wilting_shoot_tips': 0.00,
        'sparse_foliage': 0.10,
        'white_powder_coating': 0.00,
        'leaf_distortion': 0.00,
        'premature_leaf_drop': 0.05,
        'black_sooty_coating': 0.95,
        'sooty_deposit_wiped_off': 0.90,
        'yellow_discoloration': 0.20,
        'brown_leaf_margins': 0.10,
        'brown_leaf_tips': 0.05,
        'water_soaked_lesions': 0.00,
        'leaf_curling': 0.00,
    },
}

FRUIT_SYMPTOM_PROBS = {
    'Anthracnose': {
        'black_sunken_lesions': 0.90,
        'brown_patches_spreading': 0.85,
        'surface_cracks_radiating': 0.60,
        'pink_spore_masses_on_lesion': 0.55,
        'lesion_ring_pattern': 0.70,
        'soft_rot_spreading': 0.50,
        'stem_end_rot': 0.30,
        'shriveling_of_fruit': 0.40,
        'premature_fruit_drop': 0.45,
        'white_powder_on_fruit': 0.00,
        'black_sooty_deposit_on_fruit': 0.05,
        'fruit_discoloration': 0.75,
    },
    'Healthy': {
        'black_sunken_lesions': 0.00,
        'brown_patches_spreading': 0.02,
        'surface_cracks_radiating': 0.02,
        'pink_spore_masses_on_lesion': 0.00,
        'lesion_ring_pattern': 0.00,
        'soft_rot_spreading': 0.01,
        'stem_end_rot': 0.01,
        'shriveling_of_fruit': 0.02,
        'premature_fruit_drop': 0.01,
        'white_powder_on_fruit': 0.00,
        'black_sooty_deposit_on_fruit': 0.00,
        'fruit_discoloration': 0.05,
    },
}

print("Probability matrices loaded.")
print(f"  Leaf  diseases covered: {list(LEAF_SYMPTOM_PROBS.keys())}")
print(f"  Fruit diseases covered: {list(FRUIT_SYMPTOM_PROBS.keys())}")

## Synthetic Data Generation

Each sample is generated by independently sampling each symptom using its disease-specific probability, with a small noise term added to simulate real-world variability.

**Healthy class special rule:** Force at least 3 symptoms visible to simulate mild environmental stress (the model should learn that very few symptoms ≠ automatically healthy).

**Unhealthy class special rule:** Guarantee at least 1 characteristic high-probability symptom is always present (prevents degenerate samples with zero disease signal).

In [ ]:
def generate_samples(disease, symptom_list, probs_dict, n_samples,
                     noise_level=0.1, seed=42):
    """
    Generate synthetic labeled samples for a single disease class.

    Parameters
    ----------
    disease       : str   — disease name (must be a key in probs_dict)
    symptom_list  : list  — ordered list of symptom feature strings
    probs_dict    : dict  — {disease: {symptom: probability}}
    n_samples     : int   — number of samples to generate
    noise_level   : float — max uniform noise added/subtracted to each probability
    seed          : int   — base random seed for reproducibility

    Returns
    -------
    list of dict, each with keys:
        'symptoms'                : list of active symptom strings
        'disease_classification'  : str (the disease label)
    """
    rng = np.random.default_rng(seed)
    disease_probs = probs_dict[disease]
    is_healthy = (disease == 'Healthy')

    # Identify characteristic symptoms for unhealthy diseases
    # (probability >= 0.5 qualifies as characteristic)
    if not is_healthy:
        characteristic = [
            sym for sym in symptom_list
            if disease_probs.get(sym, 0.0) >= 0.50
        ]
        # Fallback: take the single highest-probability symptom
        if not characteristic:
            characteristic = [
                max(symptom_list, key=lambda s: disease_probs.get(s, 0.0))
            ]

    samples = []
    for i in range(n_samples):
        sample_seed = seed + i
        sample_rng = np.random.default_rng(sample_seed)

        active_symptoms = []
        for sym in symptom_list:
            base_prob = disease_probs.get(sym, 0.0)
            # Add uniform noise in [-noise_level, +noise_level], clamped to [0, 1]
            noise = sample_rng.uniform(-noise_level, noise_level)
            effective_prob = float(np.clip(base_prob + noise, 0.0, 1.0))
            if sample_rng.random() < effective_prob:
                active_symptoms.append(sym)

        # ── Healthy guarantee: at least 3 symptoms ────────────────────────────
        if is_healthy and len(active_symptoms) < 3:
            # Fill from symptom list randomly (minor stress symptoms)
            candidates = [s for s in symptom_list if s not in active_symptoms]
            needed = 3 - len(active_symptoms)
            chosen = sample_rng.choice(candidates,
                                       size=min(needed, len(candidates)),
                                       replace=False)
            active_symptoms.extend(chosen.tolist())

        # ── Unhealthy guarantee: at least 1 characteristic symptom ────────────
        if not is_healthy:
            has_characteristic = any(s in active_symptoms for s in characteristic)
            if not has_characteristic:
                # Force one characteristic symptom chosen at random
                forced = sample_rng.choice(characteristic)
                active_symptoms.append(forced)

        samples.append({
            'symptoms': sorted(active_symptoms),  # sorted for reproducibility
            'disease_classification': disease,
        })

    return samples


print("generate_samples() defined.")

# Quick smoke test
_test = generate_samples('Anthracnose', LEAF_SYMPTOMS, LEAF_SYMPTOM_PROBS, n_samples=3, seed=0)
for row in _test:
    print(f"  {row['disease_classification']}: {row['symptoms']}")

In [ ]:
N_SAMPLES_PER_CLASS = 150

# ── Generate leaf data ───────────────────────────────────────────────────────
leaf_samples = []
for idx, disease in enumerate(LEAF_DISEASES):
    samples = generate_samples(
        disease=disease,
        symptom_list=LEAF_SYMPTOMS,
        probs_dict=LEAF_SYMPTOM_PROBS,
        n_samples=N_SAMPLES_PER_CLASS,
        noise_level=0.1,
        seed=42 + idx * 1000,
    )
    leaf_samples.extend(samples)
    print(f"  [LEAF] {disease:15s}: {len(samples)} samples generated")

print(f"\nTotal leaf samples : {len(leaf_samples)}")

# ── Generate fruit data ──────────────────────────────────────────────────────
fruit_samples = []
for idx, disease in enumerate(FRUIT_DISEASES):
    samples = generate_samples(
        disease=disease,
        symptom_list=FRUIT_SYMPTOMS,
        probs_dict=FRUIT_SYMPTOM_PROBS,
        n_samples=N_SAMPLES_PER_CLASS,
        noise_level=0.1,
        seed=99 + idx * 1000,
    )
    fruit_samples.extend(samples)
    print(f"  [FRUIT] {disease:15s}: {len(samples)} samples generated")

print(f"\nTotal fruit samples: {len(fruit_samples)}")
print(f"Grand total        : {len(leaf_samples) + len(fruit_samples)}")

In [ ]:
# ── Build DataFrames ─────────────────────────────────────────────────────────

def samples_to_df(samples, symptom_list, disease_type_label):
    """
    Convert list-of-dicts from generate_samples() into a feature DataFrame.

    Columns:
      - one binary column per symptom (0 / 1)
      - disease_classification : str
      - disease_type           : 'leaf' | 'fruit'
      - selected_symptoms      : JSON string of active symptom list
    """
    rows = []
    for s in samples:
        active = set(s['symptoms'])
        row = {sym: (1 if sym in active else 0) for sym in symptom_list}
        row['disease_classification'] = s['disease_classification']
        row['disease_type'] = disease_type_label
        row['selected_symptoms'] = json.dumps(sorted(list(active)))
        rows.append(row)

    col_order = symptom_list + ['disease_classification', 'disease_type', 'selected_symptoms']
    return pd.DataFrame(rows, columns=col_order)


df_leaf  = samples_to_df(leaf_samples,  LEAF_SYMPTOMS,  'leaf')
df_fruit = samples_to_df(fruit_samples, FRUIT_SYMPTOMS, 'fruit')

# Combined — only shared columns are kept for the union
shared_cols = ['disease_classification', 'disease_type', 'selected_symptoms']
df_all = pd.concat(
    [df_leaf[shared_cols], df_fruit[shared_cols]],
    ignore_index=True
)

print("Leaf DataFrame shape : ", df_leaf.shape)
print("Fruit DataFrame shape: ", df_fruit.shape)
print("Combined shape       : ", df_all.shape)

# ── Save CSV files ───────────────────────────────────────────────────────────
df_leaf.to_csv('leaf_training_data.csv',  index=False)
df_fruit.to_csv('fruit_training_data.csv', index=False)
df_all.to_csv('all_training_data.csv',    index=False)
print("\nCSV files saved:")
print("  leaf_training_data.csv")
print("  fruit_training_data.csv")
print("  all_training_data.csv")

# ── Save JSON preview ────────────────────────────────────────────────────────
preview = {}

for disease in LEAF_DISEASES:
    disease_rows = [s for s in leaf_samples if s['disease_classification'] == disease]
    preview[f'leaf_{disease}'] = disease_rows[:5]

for disease in FRUIT_DISEASES:
    disease_rows = [s for s in fruit_samples if s['disease_classification'] == disease]
    preview[f'fruit_{disease}'] = disease_rows[:5]

with open('sample_data_preview.json', 'w') as f:
    json.dump(preview, f, indent=2)

print("  sample_data_preview.json")

## Data Verification

Confirm class distributions, average symptom counts, and spot-check sample rows before moving to training.

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────────
print("=" * 60)
print("LEAF CLASS DISTRIBUTION")
print("=" * 60)
leaf_counts = df_leaf['disease_classification'].value_counts()
print(leaf_counts.to_string())

print()
print("=" * 60)
print("FRUIT CLASS DISTRIBUTION")
print("=" * 60)
fruit_counts = df_fruit['disease_classification'].value_counts()
print(fruit_counts.to_string())

# ── Average symptoms per class ───────────────────────────────────────────────
print()
print("=" * 60)
print("AVERAGE ACTIVE SYMPTOMS PER LEAF CLASS")
print("=" * 60)
for disease in LEAF_DISEASES:
    subset = df_leaf[df_leaf['disease_classification'] == disease]
    # Count symptoms from binary feature columns only
    symptom_cols = LEAF_SYMPTOMS
    avg = subset[symptom_cols].sum(axis=1).mean()
    print(f"  {disease:20s}: {avg:.2f} avg symptoms")

print()
print("=" * 60)
print("AVERAGE ACTIVE SYMPTOMS PER FRUIT CLASS")
print("=" * 60)
for disease in FRUIT_DISEASES:
    subset = df_fruit[df_fruit['disease_classification'] == disease]
    symptom_cols = FRUIT_SYMPTOMS
    avg = subset[symptom_cols].sum(axis=1).mean()
    print(f"  {disease:20s}: {avg:.2f} avg symptoms")

# ── Sample row per leaf class ────────────────────────────────────────────────
print()
print("=" * 60)
print("SAMPLE ROW PER LEAF DISEASE (selected_symptoms column)")
print("=" * 60)
for disease in LEAF_DISEASES:
    row = df_leaf[df_leaf['disease_classification'] == disease].iloc[0]
    syms = json.loads(row['selected_symptoms'])
    print(f"\n  [{disease}]")
    print(f"  Symptoms ({len(syms)}): {syms}")

# ── Sample row per fruit class ───────────────────────────────────────────────
print()
print("=" * 60)
print("SAMPLE ROW PER FRUIT DISEASE (selected_symptoms column)")
print("=" * 60)
for disease in FRUIT_DISEASES:
    row = df_fruit[df_fruit['disease_classification'] == disease].iloc[0]
    syms = json.loads(row['selected_symptoms'])
    print(f"\n  [{disease}]")
    print(f"  Symptoms ({len(syms)}): {syms}")

print()
print("=" * 60)
print("Data saved to CSV files — upload these to Notebook 02 for training.")
print("=" * 60)

## Getting Real Data from MangoSense

Once the MangoSense backend has accumulated enough verified predictions with farmer-reported symptoms, replace or augment this synthetic data with real data.

**Step 1 — Check data readiness:**

```
GET /api/symptom-classifier/data-readiness/
Authorization: Bearer <admin_token>
```

This endpoint returns:
- Number of verified symptom records per disease class
- Whether the dataset meets minimum thresholds for retraining
- Class balance statistics

**Step 2 — Export training data:**

```
GET /api/symptom-classifier/export-training-data/?format=csv
Authorization: Bearer <admin_token>
```

This returns a CSV with the same column schema as `leaf_training_data.csv` and `fruit_training_data.csv` produced by this notebook. Drop it directly into **Notebook 02** in place of (or combined with) the synthetic CSVs.

**Recommended hybrid strategy:**
1. Use synthetic data (this notebook) to train an initial model
2. Deploy and collect real farmer-reported data
3. Once you have ≥ 30 real verified samples per class, mix real + synthetic (e.g. 70% real, 30% synthetic) and retrain
4. Once you have ≥ 100 real verified samples per class, drop synthetic data entirely